# Tutorial - Virtual Screening

First, let's verify that `Boltz-tools` is installed correctly by running:

In [ ]:
!boltz-lab --version

If the installation was successful, this command will display the installed version number of `Boltz-tools`.

If you encounter any errors, refer to the installation guide in the `README.md` file to troubleshoot and complete the setup.

> **Note:**  
> - When running these commands in a regular command-line interface (CLI), remove the `!` at the beginning of the command.
> - If you're copying commands from a Jupyter Notebook that use backslashes (`\`) to split long commands across multiple lines, remove the backslashes and write the command on a single line in the CLI.

## Command and Input Files

The `screen` subcommand in `Boltz-tools` is used to perform virtual screening against a library of compounds. To use it effectively, it’s important to understand both the command structure and the required input files.

### 1. Command 

To understand how to use the screening workflow, start by viewing the command help:

In [ ]:
!boltz-lab screen --help

### Key Arguments and Options

| **Flag/Argument** | **Description** |
|-------------------|-----------------|
| `-w, --wrk_dir` | Working directory. Defaults to the current directory if not specified. |
| `-s, --system_path` | Path to the **system YAML file**, which defines the system (e.g., protein, receptor, or other setup details). |
| `-b, --boltz_options_path` | Path to the **Boltz options YAML file** specifying screening settings and parameters. |
| `-v, --variable` | Comma-separated nested path to update variables in the system YAML (e.g., `sequences,1,ligand,smiles`). |
| `-c, --variable_csv` | Path to a **CSV file** containing variables (e.g., ligands). Requires `--col_variable` and `--col_id`. |
| `--col_variable` | Column in the **CSV file** containing the variable, such as `SMILES` or `CCD` identifiers. |
| `--col_id` | Column in the **CSV file** containing the compound IDs. |
| `-s, --variable_sdf` | Path to an **SDF file** containing variables. Requires `--property_id`. |
| `--property_id` | Property in the **SDF file** used for compound IDs. |
| `--generate_conformers` | Optionally generate `2D` or `3D` conformers for input compounds (**works only for SMILES input**). |
| `--merge_data` | Optional: Merge extra data fields from input CSV or SDF into the output. |
| `-d, --debug` | Enables debug logging for troubleshooting. |


### Required Input Files

To run `boltz-lab screen`, you need the following files:

- A **system file** (`.yaml`) describing the biological system and initial settings  
- An **options file** (`.yaml`) specifying screening parameters and workflow settings  
- **Either** a **screening file** in **SDF** (`.sdf`) **or** **CSV** (`.csv`) format containing the ligands to be screened  

Let's go through the imput files:


#### System File

A minimal `system.yaml` file might look like this:

In [ ]:
!cat ../examples/system_screen.yaml

In this setup, the ligand does not yet have a SMILES string.
To specify the SMILES dynamically during screening, use the `-v` option in the command:
 - `-v sequences,1,ligand,smiles` tells `Boltz-tool`s to insert or update the `SMILES` value for the ligand entry.
 - If the system file already contained a `SMILES` value, this command would overwrite it with the values from the screening file (`.csv` or `.sdf`).

In Boltz-tools, the `-v` option allows you to **update any variable** in the system YAML dynamically, not just ligand SMILES.  

For example, you could modify:  
- Ligand SMILES  
- Protein sequences  
- Any other nested variable defined in the system  

The system file itself is also flexible: **any valid system YAML** that can be uploaded to Boltz-tools can be used here.  
This allows you to screen different ligands, protein variants, or combinations thereof, without modifying the original system file manually.  


#### Options File

The **options file** allows you to customize the behavior of `Boltz` during a screening run.  
To see the full list of available settings, you can run:

In [ ]:
!boltz predict --help

This will display all the command-line options that can also be set or adjusted in an options YAML file.

You specify the options file in the command using the -b flag.

Here is a basic example of an options file (`options.yaml`) where only the cache directory is set:

In [ ]:
!cat ../examples/options.yaml

This file can be extended to include other Boltz parameters, such as sampling_steps, diffusion_samples and use_potentials.

#### Screening File (CSV)

A **screening file** provides the ligands/peptides/proteins to be evaluated during virtual screening.  

Here’s an example CSV file:

In [ ]:
!cat ../examples/ligand_screen.csv

- `SMILES` is the variable column specified with `--col_variable`.
- `Name` is the compound ID column specified with `--col_id`.
- `pIC50` is an additional property we want to carry forward into the results for post-analysis, so we add it using `--merge_data`.

#### Screening File (SDF)

This same file can also be provided in SDF, which allows the usage of more advanced options using CCD codes. See `advanced tutorial`.

In [ ]:
#TODO: Add example with SDF (exact same as CSV)
#!cat ../examples/ligand_screen.sdf

# Script

Run virtual screening on EGFR kinase domain from SMILES list:

In [ ]:
!boltz-lab screen \
    -w ./tmp/tutorial_screen_smiles \
    -s ../examples/system.yaml \
    -b ../examples/options.yaml \
    -v sequences,1,ligand,smiles \
    -c ../examples/ligand_screen.csv \
    --col_variable SMILES \
    --col_id Name \
    --merge_data pIC50

In [ ]:
!boltz-lab screen \
    -w ./tmp/tutorial_screen_sdf_2d \
    -s ../examples/system.yaml \
    -b ../examples/options.yaml \
    -v sequences,1,ligand,ccd \
    -c ../examples/ligand_screen.csv \
    --col_variable SMILES \
    --col_id Name \
    --merge_data pIC50 \
    --generate_conformers 2D

In [ ]:
!boltz-lab screen \
    -w ./tmp/tutorial_screen_sdf_3d \
    -s ../examples/system.yaml \
    -b ../examples/options.yaml \
    -v sequences,1,ligand,ccd \
    -c ../examples/ligand_screen.csv \
    --col_variable SMILES \
    --col_id Name \
    --merge_data pIC50 \
    --generate_conformers 3D

# ---------------------

If you want to remove the temporary directory or cache after running the tutorial, uncomment the line below:

In [ ]:
!rm -rf ./tmp
#!rm -rf ./cache
!rm -rf ./cache/.boltz/mols/caffe.pkl
!rm -rf ./cache/.boltz/mols/erlot.pkl
!rm -rf ./cache/.boltz/mols/lapat.pkl
!rm -rf ./cache/.boltz/mols/gefit.pkl

!rm -rf ../examples/tmp/ligand_quick.sdf